# Spark Processing - Transformations & Aggregations

This notebook performs Spark transformations and aggregations on the Olist e-commerce data.

**Input:** `data/processed/orders_core.parquet`

**Outputs:**
- `customer_spending.parquet` - Customer lifetime value and spending patterns
- `top_categories.parquet` - Category performance metrics
- `monthly_trends.parquet` - Monthly sales trends
- `customer_segments.parquet` - Customer segmentation based on behavior

## 1. Setup and Initialize Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum, avg, min, max, round,
    to_date, year, month, date_format, unix_timestamp,
    ntile, dense_rank, row_number, desc, asc,
    coalesce, when, regexp_extract
)
from pyspark.sql.window import Window
import os

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Olist-Spark-Processing") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("INFO")

print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.appName}")

## 2. Load Input Data

In [ ]:
# Read the processed orders data
orders_df = spark.read.parquet("../data/processed/orders_core.parquet")

print(f"Orders DataFrame shape: {orders_df.count()} rows")
print("\nSchema:")
orders_df.printSchema()

print("\nFirst few rows:")
orders_df.show(5, truncate=False)

## 3. Customer Spending Analysis

In [ ]:
# Customer spending: total spend, order count, average order value, etc.
customer_spending = orders_df.groupBy("customer_id", "customer_city", "customer_state") \
    .agg(
        count("order_id").alias("order_count"),
        round(sum("order_value"), 2).alias("total_spending"),
        round(avg("order_value"), 2).alias("avg_order_value"),
        round(min("order_value"), 2).alias("min_order_value"),
        round(max("order_value"), 2).alias("max_order_value"),
        count(when(col("payment_value") == 0, 1)).alias("zero_payment_orders")
    ) \
    .withColumn(
        "customer_segment",
        when(col("total_spending") >= 1000, "VIP")
        .when(col("total_spending") >= 500, "Premium")
        .when(col("total_spending") >= 100, "Regular")
        .otherwise("Occasional")
    ) \
    .sort(col("total_spending").desc())

print("Customer Spending Summary:")
customer_spending.show(10, truncate=False)

print(f"\nTotal customers: {customer_spending.count()}")

## 4. Top Categories Analysis

In [ ]:
# Category performance: revenue, order count, avg value, etc.
top_categories = orders_df \
    .filter(col("product_category_name").isNotNull()) \
    .groupBy("product_category_name") \
    .agg(
        count("order_id").alias("total_orders"),
        count(col("product_id")).alias("total_items_sold"),
        round(sum("order_value"), 2).alias("total_revenue"),
        round(avg("order_value"), 2).alias("avg_order_value"),
        round(avg(col("product_price")), 2).alias("avg_product_price"),
        count(when(col("review_score").isNotNull(), 1)).alias("reviewed_orders"),
        round(avg("review_score"), 2).alias("avg_review_score")
    ) \
    .withColumn(
        "review_rate",
        round(col("reviewed_orders") / col("total_orders") * 100, 2)
    ) \
    .sort(col("total_revenue").desc())

print("Top Categories by Revenue:")
top_categories.show(15, truncate=False)

print(f"\nTotal unique categories: {top_categories.count()}")

## 5. Monthly Trends Analysis

In [ ]:
# Convert order dates and extract month/year
orders_with_dates = orders_df \
    .withColumn("order_purchase_timestamp", to_date(col("order_purchase_timestamp"))) \
    .withColumn("year_month", date_format(col("order_purchase_timestamp"), "yyyy-MM")) \
    .withColumn("year", year(col("order_purchase_timestamp"))) \
    .withColumn("month", month(col("order_purchase_timestamp")))

# Monthly trends: orders, revenue, avg order value
monthly_trends = orders_with_dates.groupBy("year_month", "year", "month") \
    .agg(
        count("order_id").alias("order_count"),
        count(col("customer_id")).alias("unique_customers"),
        round(sum("order_value"), 2).alias("total_revenue"),
        round(avg("order_value"), 2).alias("avg_order_value"),
        count(col("product_id")).alias("items_sold"),
        count(when(col("order_status") == "delivered", 1)).alias("delivered_orders"),
        count(when(col("order_status") == "canceled", 1)).alias("canceled_orders")
    ) \
    .withColumn(
        "delivery_rate",
        round(col("delivered_orders") / col("order_count") * 100, 2)
    ) \
    .sort("year", "month")

print("Monthly Sales Trends:")
monthly_trends.show(20, truncate=False)

## 6. Customer Segmentation (RFM Analysis)

In [ ]:
# Calculate RFM (Recency, Frequency, Monetary) metrics
from pyspark.sql.functions import max as spark_max, datediff, current_date

# Get max date for recency calculation
max_date = orders_with_dates.agg(spark_max("order_purchase_timestamp")).collect()[0][0]
print(f"Max date in dataset: {max_date}")

# RFM Analysis
rfm_data = orders_with_dates.groupBy("customer_id") \
    .agg(
        datediff(lit(max_date), spark_max("order_purchase_timestamp")).alias("recency"),
        count("order_id").alias("frequency"),
        round(sum("order_value"), 2).alias("monetary_value")
    )

# Define window for quartile ranking
window_spec = Window.partitionBy().orderBy(col("recency"))

customer_segments = rfm_data \
    .withColumn("r_score", ntile(4).over(window_spec.orderBy("recency"))) \
    .withColumn("f_score", ntile(4).over(window_spec.orderBy("frequency"))) \
    .withColumn("m_score", ntile(4).over(window_spec.orderBy("monetary_value"))) \
    .withColumn(
        "segment",
        when((col("r_score") >= 3) & (col("f_score") >= 3) & (col("m_score") >= 3), "Champions")
        .when((col("r_score") >= 3) & (col("f_score") >= 2), "Loyal Customers")
        .when((col("r_score") < 2) & (col("f_score") >= 3), "At Risk")
        .when((col("r_score") >= 2) & (col("f_score") >= 2) & (col("m_score") >= 2), "Potential")
        .otherwise("Need Attention")
    ) \
    .sort(col("monetary_value").desc())

print("Customer Segmentation (RFM):")
customer_segments.show(15, truncate=False)

print("\nSegment Distribution:")
customer_segments.groupBy("segment").count().sort("count", ascending=False).show()

## 7. Save Output Files

In [ ]:
from pyspark.sql.functions import lit

# Create output directory if it doesn't exist
output_path = "../data/processed"

# Save customer spending
customer_spending.coalesce(1).write.mode("overwrite").parquet(f"{output_path}/customer_spending.parquet")
print("✓ Saved customer_spending.parquet")

# Save top categories
top_categories.coalesce(1).write.mode("overwrite").parquet(f"{output_path}/top_categories.parquet")
print("✓ Saved top_categories.parquet")

# Save monthly trends
monthly_trends.coalesce(1).write.mode("overwrite").parquet(f"{output_path}/monthly_trends.parquet")
print("✓ Saved monthly_trends.parquet")

# Save customer segments
customer_segments.coalesce(1).write.mode("overwrite").parquet(f"{output_path}/customer_segments.parquet")
print("✓ Saved customer_segments.parquet")

print(f"\nAll outputs saved to {output_path}/")

## 8. Summary Statistics

In [ ]:
print("=" * 60)
print("SPARK PROCESSING SUMMARY")
print("=" * 60)

print(f"\nTotal Orders Processed: {orders_df.count():,}")
print(f"Total Customers: {customer_spending.count():,}")
print(f"Unique Categories: {top_categories.count():,}")
print(f"Date Range: {orders_with_dates.agg(spark_max('order_purchase_timestamp')).collect()[0][0]}")

print("\nCustomer Segment Distribution:")
customer_spending.groupBy("customer_segment").count().sort("count", ascending=False).show()

print("\nRFM Segment Distribution:")
customer_segments.groupBy("segment").count().sort("count", ascending=False).show()

print("\n" + "=" * 60)
print("Processing Complete! Ready for MongoDB ingestion.")
print("=" * 60)